In [15]:
#Import bibliotecas
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, current_timestamp, from_json, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [2]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [5]:
#Iniciando sessão spark
spark = SparkSession.builder \
    .appName("bronze-layer") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e23043e3-c705-40af-91a4-1cde7c692326;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
:: resolution report :: resolve 296ms :: artifacts dl 13ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 from central in [default]
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | numbe

In [32]:
!ls /tmp/warehouse/bronze/transacoes

data  metadata


In [6]:
#Variaveis de conexao ao ambiente postgree
conexao = {
           "url": "jdbc:postgresql://postgres:5432/pipeline_transacoes",
           "user": "postgres",
           "password": "1234",
           "driver": "org.postgresql.Driver"
}

In [7]:
#Dados vindo da API
df_raw = spark.read.jdbc(
    url=conexao["url"],
    table="staging.transacoes_raw",  # tabela no PostgreSQL
    properties={
        "user": conexao["user"],
        "password": conexao["password"],
        "driver": conexao["driver"]
    }
)

df_raw.select("payload").show(truncate=True)

+--------------------+
|             payload|
+--------------------+
|{"valor": 3716.46...|
|{"valor": 2860.04...|
|{"valor": 4207.49...|
|{"valor": 429.82,...|
|{"valor": 787.63,...|
|{"valor": 1772.99...|
|{"valor": 70.52, ...|
|{"valor": 3214.31...|
|{"valor": 4886.47...|
|{"valor": 1886.66...|
|{"valor": 3692.48...|
|{"valor": 1648.44...|
|{"valor": null, "...|
|{"valor": null, "...|
|{"valor": 4551.95...|
|{"valor": 138.62,...|
|{"valor": 1923.8,...|
|{"valor": 4191.04...|
|{"valor": 1987.56...|
|{"valor": 1441.97...|
+--------------------+
only showing top 20 rows



In [8]:
#Criando tabela bronze se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

#Definir schema do payload
schema_payload = StructType([
    StructField("valor", DoubleType(), True),
    StructField("produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("metodo_pagamento", StringType(), True),
])

# Transformar DF raw
df_bronze = (
    df_raw
    .withColumn("data_ingestao", current_timestamp())
    .withColumn("data_particao", to_date(col("data_ingestao")))
    .withColumn("payload_struct", from_json(col("payload"), schema_payload))
    .select(
        "id",
        "payload_struct.*",
        "data_ingestao",
        "data_particao"
    )
)                           
df_bronze.writeTo("lakehouse.bronze.transacoes") \
         .using("iceberg") \
         .partitionedBy("data_particao") \
         .createOrReplace()

In [9]:
#Select tabela transacoes na bronze
spark.sql("SELECT * FROM lakehouse.bronze.transacoes LIMIT 5").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  1|3716.46|camiseta|  vestuario|        11|         1|             pix|2026-03-30 01:55:...|   2026-03-30|
|  2|2860.04|   tenis|  vestuario|         1|         1|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
|  3|4207.49| celular|eletronicos|        80|         2|          boleto|2026-03-30 01:55:...|   2026-03-30|
|  4| 429.82|   tenis|  vestuario|        43|         3|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
|  5| 787.63|    fone|eletronicos|        19|         1|             pix|2026-03-30 01:55:...|   2026-03-30|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+



In [10]:
#Validando estrutura tabela bronze
spark.sql("Describe lakehouse.bronze.transacoes").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|                  id|      int|   NULL|
|               valor|   double|   NULL|
|             produto|   string|   NULL|
|           categoria|   string|   NULL|
|          cliente_id|      int|   NULL|
|          quantidade|      int|   NULL|
|    metodo_pagamento|   string|   NULL|
|       data_ingestao|timestamp|   NULL|
|       data_particao|     date|   NULL|
|# Partition Infor...|         |       |
|          # col_name|data_type|comment|
|       data_particao|     date|   NULL|
+--------------------+---------+-------+



In [11]:
#Validando a existencia de dados nulos na bronze
from pyspark.sql.functions import col, sum
df_bronze.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_bronze.columns
]).show()

+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
|  0|   11|      0|        0|         0|         0|               0|            0|            0|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+



In [21]:
#Verificando dados nulos no campo VALOR
spark.sql("SELECT * FROM lakehouse.bronze.transacoes where valor is null").show()

+---+-----+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-----+--------+-----------+----------+----------+----------------+--------------------+-------------+
| 13| NULL|   tenis|  vestuario|        71|         1|             pix|2026-03-30 01:55:...|   2026-03-30|
| 14| NULL|   tenis|  vestuario|        47|         1|          boleto|2026-03-30 01:55:...|   2026-03-30|
| 27| NULL|    fone|eletronicos|         3|         2|             pix|2026-03-30 01:55:...|   2026-03-30|
| 30| NULL| celular|eletronicos|        43|         1|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
| 33| NULL|notebook|eletronicos|        35|         2|             pix|2026-03-30 01:55:...|   2026-03-30|
| 34| NULL| celular|eletronicos|        48|         2|          boleto|2026-03-30 01:55:...|   2026-03-30|
| 40| NULL|camiseta|  vestuario|     

In [14]:
#Validando se a duplicata para possivel tratamento
df_bronze.groupBy("id", "data_ingestao") \
         .agg(count("*").alias("qtd")) \
         .filter(col("qtd") > 1) \
         .show()

+---+-------------+---+
| id|data_ingestao|qtd|
+---+-------------+---+
+---+-------------+---+



In [20]:
#Gerando tabela transacao SILVER com base na bronze porem com tratamento de campos null

# Ler Bronze
df_bronze = spark.read.table("lakehouse.bronze.transacoes")

#Tratamento valor null por 0
df_curated = (
    df_bronze.fillna({"valor": 0})
)

# Criar database Curated se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.silver")

# Salvar Curated em Delta Lake, particionado por data
df_curated.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()


spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  1|3716.46|camiseta|  vestuario|        11|         1|             pix|2026-03-30 01:55:...|   2026-03-30|
|  2|2860.04|   tenis|  vestuario|         1|         1|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
|  3|4207.49| celular|eletronicos|        80|         2|          boleto|2026-03-30 01:55:...|   2026-03-30|
|  4| 429.82|   tenis|  vestuario|        43|         3|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
|  5| 787.63|    fone|eletronicos|        19|         1|             pix|2026-03-30 01:55:...|   2026-03-30|
|  6|1772.99|notebook|eletronicos|        95|         3|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
|  7|  70.52|camise

In [36]:
#Atualização na tabela silver (ADD COLUMNS)
df_add = df_curated.withColumn("preco_unitario", (col("valor") / col("quantidade")))
df_add.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()

In [38]:
#Validando após alteração
spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|3716.46|camiseta|  vestuario|        11|         1|             pix|2026-03-30 01:55:...|   2026-03-30|           3716.46|
|  2|2860.04|   tenis|  vestuario|         1|         1|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|           2860.04|
|  3|4207.49| celular|eletronicos|        80|         2|          boleto|2026-03-30 01:55:...|   2026-03-30|          2103.745|
|  4| 429.82|   tenis|  vestuario|        43|         3|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|143.27333333333334|
|  5| 787.63|    fone|eletronicos|        19|         1|             pix|2026-03-30 01:55:...|   2026-03